# 📊 Exploratory Data Analysis (EDA) — House Price Prediction

This notebook performs the Exploratory Data Analysis (EDA) stage for the Ames, Iowa Housing dataset. We focus on our target variable (`SalePrice`) and three key features:
1. Above-Grade Living Area (`GrLivArea`)
2. Bedrooms Above Ground (`BedroomAbvGr`)
3. Bathrooms (Full and Half baths combined into `TotalBathrooms`)

### Objectives:
- Understand dataset dimensions and types
- Inspect and handle missing values
- Visualize the target variable's distribution
- Analyze correlations between variables
- Inspect features for outliers

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries successfully imported!")

In [2]:
# Load training dataset
train_df = pd.read_csv("../data/train.csv")
print(f"Dataset successfully loaded. Shape: {train_df.shape}")

In [3]:
# Shape & general info
key_cols = ["GrLivArea", "BedroomAbvGr", "FullBath", "HalfBath", "SalePrice"]
print("--- Data Types & Non-Null Counts ---")
print(train_df[key_cols].info())

print("\n--- Missing Values ---")
print(train_df[key_cols].isnull().sum())

In [4]:
# Summary statistics
print("--- Summary Statistics ---")
print(train_df[key_cols].describe())

In [5]:
# Target Distribution Plot
plt.figure(figsize=(10, 6))
sns.histplot(train_df["SalePrice"], kde=True, color="#1a73e8", bins=40)
plt.title("Distribution of SalePrice (Target Variable)", fontsize=14, fontweight="bold")
plt.xlabel("Sale Price ($)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.axvline(train_df["SalePrice"].mean(), color="red", linestyle="--", label=f"Mean: ${train_df['SalePrice'].mean():,.2f}")
plt.axvline(train_df["SalePrice"].median(), color="green", linestyle="-", label=f"Median: ${train_df['SalePrice'].median():,.2f}")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/target_distribution.png", dpi=150)
plt.show()

In [6]:
# Pre-calculate TotalBathrooms to analyze its correlation
train_df["TotalBathrooms"] = train_df["FullBath"] + 0.5 * train_df["HalfBath"]
correlation_cols = ["GrLivArea", "BedroomAbvGr", "TotalBathrooms", "SalePrice"]
corr_matrix = train_df[correlation_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".3f", linewidths=0.5, annot_kws={"size": 12})
plt.title("Correlation Heatmap of Key Features & Target", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/correlation_heatmap.png", dpi=150)
plt.show()

print("--- Correlation with SalePrice ---")
print(corr_matrix["SalePrice"].sort_values(ascending=False))

In [7]:
# Outlier inspection on Living Area vs Sale Price
plt.figure(figsize=(10, 6))
sns.scatterplot(data=train_df, x="GrLivArea", y="SalePrice", alpha=0.7, color="#34a853")
plt.title("Outlier Inspection: GrLivArea vs SalePrice", fontsize=14, fontweight="bold")
plt.xlabel("Above-Grade Living Area (sqft)", fontsize=12)
plt.ylabel("Sale Price ($)", fontsize=12)

# Annotate known outliers (large area, low price)
outliers = train_df[(train_df["GrLivArea"] > 4000) & (train_df["SalePrice"] < 300000)]
for idx, row in outliers.iterrows():
    plt.annotate(f"Outlier (Id {row['Id']})", 
                 xy=(row['GrLivArea'], row['SalePrice']),
                 xytext=(row['GrLivArea'] + 100, row['SalePrice'] - 50000),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6))

plt.tight_layout()
plt.savefig("../outputs/outlier_inspection.png", dpi=150)
plt.show()